# 57 · Is the "beyond S1/S2/S3 steps" signal within-segment biology or label–coordinate discordance?

Notebook 37's pathway test of smooth position **beyond S1/S2/S3 steps** (`T_position_given_segments`)
calls 79 pathways on the scFates coordinate. The steps are the reviewed segment **labels**, and the
smooth term is a spline along the **coordinate**. If labels and coordinate disagree for some
structures, the spline can explain variance that the label steps miss even when expression only
steps between segments. Two artefacts are possible:

- **Label noise.** A structure labelled S1 that is really S2 carries S2 expression and sits at an
  S2-like coordinate. Test A (beyond **label** steps) is exposed to this.
- **Coordinate noise.** A true step at the labelled boundary looks like a ramp along a noisy
  coordinate. Test B (beyond **coordinate-threshold** steps) is exposed to this.

This notebook measures both with semi-synthetic data on the real structures, labels and
coordinate. **Arm A** permutes expression rows within specimen × coordinate-threshold segment, so
expression depends only on the coordinate segment (truth = coordinate steps, labels noisy).
**Arm B** permutes rows within specimen × label, so expression depends only on the label (truth =
label steps, coordinate noisy). Row permutation keeps every gene's segment means and the
correlation between genes, and it removes all within-segment structure.

The coordinate is a parameter (`--coordinate`, `--coordinate-column`, `--coordinate-label`, or
`PSEUDOSPACE_PT_COORDINATE[_COLUMN|_LABEL]`). It runs unchanged on SCF13, DPT13, SCF13-ED and
workstream A's coordinate. The protocol was frozen in workstream B's phase-1 plan (section 8).

Cohort: 2 male control mice and 2 cortex sections from 1 male human donor. Everything is
descriptive within the cohort.

## 1 · Folders and pre-specified rules

| Rule | Value |
|---|---|
| Inputs | notebook 37's construction on the supplied coordinate: PT structures in every specimen's 1st–99th percentile interval, 2% equal-specimen detection, pathways with 10–300 tested genes, 3-bin covariate strata |
| Coordinate-threshold segments | per specimen, the two cut points that best reproduce its reviewed S1/S2/S3 labels (exhaustive search, ≤ 600 candidates) |
| Statistics | test A = species × spline beyond species × **label** steps; test B = the same beyond species × **coordinate-threshold** steps; 6-df spline; notebook 31's weights and specimen contrasts |
| Pathway call | joint test (matched rank-AUC z ÷ √VIF, 9,999 matched sets, seed 12), positive effect, BH q ≤ 0.05; VIF from within-specimen residual correlations of the full species model (notebook 37) |
| Semi-synthetic arms | 10 replicates per arm, seeds 5701–5710 (arm A) and 5801–5810 (arm B); the observed VIF and strata are reused |
| **Primary metric** | median species test-A calls in arm A ÷ observed test-A calls |
| **Decision** | arm-A median ≥ 50% of observed → "beyond label steps" is reported as label–coordinate discordance and supports no within-segment claim; < 20% → the label-noise explanation is rejected; otherwise partly explained |
| Within-segment candidate | observed test A **and** test B calls; across coordinates (section 7): called on SCF13 and on at least one of the other coordinates run so far |

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
parser.add_argument('--coordinate', type=Path, help='CSV with structure or structure_id and a position column')
parser.add_argument('--coordinate-column')
parser.add_argument('--coordinate-label')
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
coordinate_file = Path(args.coordinate or os.environ.get('PSEUDOSPACE_PT_COORDINATE')
                       or results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv').expanduser()
COORDINATE_COLUMN = args.coordinate_column or os.environ.get('PSEUDOSPACE_PT_COORDINATE_COLUMN') or 'position'
COORDINATE_LABEL = args.coordinate_label or os.environ.get('PSEUDOSPACE_PT_COORDINATE_LABEL') or 'scf13'
upstream13 = results_root / 'minimal_pt_scfates'
output = results_root / 'pt_pathway_continuous' / '57_beyond_steps' / COORDINATE_LABEL
tables = output / 'tables'
tables.mkdir(parents=True, exist_ok=True)
for path in (coordinate_file, upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv'):
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '57.beyond_steps.1'  # Bump when a cached calculation changes.
ALPHA, N_NULL, SEED, BASIS_DF = .05, 9999, 12, 6
N_REPLICATES = 10
ARM_SEEDS = {'A': 5701, 'B': 5801}
DECISION_HIGH, DECISION_LOW = .5, .2
# Notebook 37 folders for the reproduction guard, by coordinate label.
NB37_RUN = {'scf13': 'scfates', 'dpt13': 'dpt13', 'scf13_ed': 'scf13_ed'}
print('Coordinate:', coordinate_file, f'[{COORDINATE_COLUMN}] as', COORDINATE_LABEL)
print('Results folder:', output)

## 2 · Structures, coordinate, expression, genes and pathways

In [ ]:
import glob

import numpy as np
import pandas as pd
from IPython.display import display

from pseudospace import continuous_descriptors as descriptors_module
from pseudospace import coordinate_inputs as inputs_module
from pseudospace import pathway_calibration as calibration_module
from pseudospace.continuous_descriptors import blockwise_permutation, coordinate_segments
from pseudospace.coordinate_inputs import pt_inputs, read_coordinate
from pseudospace.pathway_calibration import balanced_partitions, contrast_designs, nested_partial_f
from pseudospace.pathway_decoys import joint_matched_test
from pseudospace.pathway_remodeling import (correlation_adjusted_rank_tests, fit_nested_trajectories,
                                            matched_pathway_tests, residual_pathway_correlations)
from pseudospace.stage_cache import cached_frame, cached_payload, digest
from pseudospace.stats_gam import gam_internal_knots

coordinate = read_coordinate(coordinate_file, COORDINATE_COLUMN)
inputs = pt_inputs(results_root, data_root, coordinate)
position, human, specimen, segment = (inputs[k] for k in ('position', 'human', 'specimen', 'segment'))
genes, gene_sets, strata = inputs['genes'], inputs['gene_sets'], inputs['strata']
Y = inputs['Y'].toarray()
display(inputs['obs'].groupby(['comparison_species', 'sample', 'segment_class'], observed=True).size()
        .unstack().rename_axis(columns=None))
print(f'{len(position):,} structures · {len(genes):,} genes · {len(gene_sets):,} pathways')

cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(descriptors_module.__file__), Path(inputs_module.__file__),
               Path(calibration_module.__file__)])
input_key = {'Y': digest(inputs['Y']), 'position': position, 'specimen': specimen, 'segment': segment,
             'genes': genes}

## 3 · Coordinate-threshold segments

Label agreement is the share of a specimen's structures whose reviewed label matches the segment
implied by the two best cut points. The rest are the structures on which labels and coordinate
disagree.

In [ ]:
coord_segment, thresholds = coordinate_segments(position, segment, specimen)
thresholds['species'] = np.where(thresholds.specimen.str.startswith('HUK'), 'human', 'mouse')
thresholds.to_csv(tables / 'coordinate_thresholds.csv', index=False)
display(thresholds.round(3))
disagreement = pd.crosstab(segment, coord_segment, rownames=['label'], colnames=['coordinate segment'])
display(disagreement)

## 4 · Observed statistics, variance inflation and joint pathway tests

Gene scores for the species split and both relabelings (notebook 31's designs; a relabeled
model carries the species shape as a nuisance term). The VIF comes from the full species model's
within-specimen residual correlations, exactly as in notebook 37.

In [ ]:
knots = gam_internal_knots(position, basis_df=BASIS_DF)
partitions = balanced_partitions(specimen, np.where(human, 'human', 'mouse'))


def beyond_step_scores(values, label):
    if label == 'species':
        group, nuisance = human.astype(float), None
    else:
        group, nuisance = np.isin(specimen, partitions[label]).astype(float), human.astype(float)
    out = {}
    for name, segs in (('test_A_label_steps', segment), ('test_B_coordinate_steps', coord_segment)):
        designs, weights = contrast_designs(position, group, specimen, knots, segments=segs, nuisance_shape=nuisance)
        scores = nested_partial_f(values, designs, weights, {'T_position_given_segments': ('discrete', 'union'),
                                                             'T_spatial': ('level', 'full')})
        out[name] = scores['T_position_given_segments']
        out['T_spatial'] = scores['T_spatial']
    return out


observed_scores = {label: pd.DataFrame(cached_payload('scores_' + label.replace(':', '_').replace('+', '_'),
                   lambda label=label: beyond_step_scores(Y, label), root=cache,
                   params={'basis_df': BASIS_DF, 'label': label}, inputs=input_key, code=code), index=genes)
                   for label in partitions}
run37 = results_root / 'pt_pathway_final' / NB37_RUN.get(COORDINATE_LABEL, '-') / 'stage_cache'
saved37 = sorted(glob.glob(str(run37 / 'gene_scores_species__*.npz')))
if len(saved37) == 1:  # Guard: test A reproduces notebook 37's statistic on the same coordinate.
    reference = np.load(saved37[0])
    if len(reference['T_position_given_segments']) == len(genes):
        np.testing.assert_allclose(observed_scores['species'].test_A_label_steps,
                                   reference['T_position_given_segments'], rtol=1e-6, atol=1e-8)
        print('Guard passed: test A equals notebook 37 T_position_given_segments on this coordinate.')


def species_vif():
    fit = fit_nested_trajectories(inputs['Y'], position, human, specimen, inputs['grid'], basis_df=BASIS_DF,
                                  return_residuals=True)
    correlations = residual_pathway_correlations(fit['residuals'], specimen, genes, gene_sets)
    camera = correlation_adjusted_rank_tests(pd.Series(fit['T_spatial'], index=genes), gene_sets, correlations)
    return camera[['pathway_id', 'variance_inflation']]


vif = cached_frame('species_vif', species_vif, root=cache, params={'basis_df': BASIS_DF},
                   inputs={**input_key, 'sets': gene_sets}, code=code).set_index('pathway_id').variance_inflation
saved_table = results_root / 'pt_pathway_final' / NB37_RUN.get(COORDINATE_LABEL, '-') / 'tables' / 'table_S2_pathways.csv'
if saved_table.is_file():  # Guard: same VIF as notebook 37 on the same coordinate.
    reference = pd.read_csv(saved_table).set_index('pathway_id').vif
    if set(reference.index) == set(vif.index):
        np.testing.assert_allclose(vif.reindex(reference.index), reference, rtol=1e-6)
        print('Guard passed: VIF equals notebook 37 on this coordinate.')
print(f'Median VIF {vif.median():.2f}')


def joint_calls(scores, stage_label):
    frame = cached_frame('matched_' + stage_label, lambda: matched_pathway_tests(scores, gene_sets, strata,
                         n_null=N_NULL, seed=SEED), root=cache, params={'n_null': N_NULL, 'seed': SEED},
                         inputs={'scores': scores, 'sets': gene_sets, 'strata': strata}, code=code)
    return joint_matched_test(frame, vif, keys=('statistic',))


observed = pd.concat([joint_calls(frame, 'observed_' + label.replace(':', '_').replace('+', '_')).assign(partition=label)
                      for label, frame in observed_scores.items()], ignore_index=True)
observed['called'] = observed.effect.gt(0) & observed.q_joint.le(ALPHA)
observed.to_csv(tables / 'observed_joint_tests.csv', index=False)
observed_counts = observed.pivot_table(index='statistic', columns='partition', values='called', aggfunc='sum')
display(observed_counts)

## 5 · Semi-synthetic arms

Arm A permutes expression rows within specimen × coordinate-threshold segment, and arm B within
specimen × label. Positions and labels stay with their structures. Each replicate reruns both
tests on the species split with the observed VIF and strata.

In [ ]:
blocks = {'A': np.array([f'{a}|{b}' for a, b in zip(specimen, coord_segment)]),
          'B': np.array([f'{a}|{b}' for a, b in zip(specimen, segment)])}


def arm_replicate(arm, rep):
    index = blockwise_permutation(blocks[arm], np.random.default_rng(ARM_SEEDS[arm] + rep))
    scores = pd.DataFrame(beyond_step_scores(Y[index], 'species'), index=genes)
    frame = matched_pathway_tests(scores, gene_sets, strata, n_null=N_NULL, seed=SEED)
    tests = joint_matched_test(frame, vif, keys=('statistic',))
    tests['called'] = tests.effect.gt(0) & tests.q_joint.le(ALPHA)
    return tests[['pathway_id', 'statistic', 'z_joint', 'q_joint', 'called']].assign(arm=arm, replicate=rep)


simulated = pd.concat([cached_frame(f'arm_{arm}_{rep}', lambda arm=arm, rep=rep: arm_replicate(arm, rep), root=cache,
                                    params={'arm': arm, 'rep': rep, 'seed': ARM_SEEDS[arm], 'n_null': N_NULL},
                                    inputs={**input_key, 'vif': vif, 'coord_segment': coord_segment}, code=code)
                       for arm in ('A', 'B') for rep in range(1, N_REPLICATES + 1)], ignore_index=True)
simulated.to_csv(tables / 'semi_synthetic_tests.csv', index=False)
arm_counts = simulated.groupby(['arm', 'statistic', 'replicate']).called.sum().unstack('replicate')
display(arm_counts)

## 6 · Decision

The primary metric is the arm-A median of test-A calls divided by the observed test-A calls.
Arm B gives the matching null for test B (coordinate noise). The per-pathway rows compare each
observed joint z with the largest z it reached in either arm.

In [ ]:
obs_species = observed[observed.partition.eq('species')]
n_obs = {stat: int(obs_species[obs_species.statistic.eq(stat)].called.sum())
         for stat in ('test_A_label_steps', 'test_B_coordinate_steps', 'T_spatial')}
median_arm = simulated.groupby(['arm', 'statistic', 'replicate']).called.sum().groupby(['arm', 'statistic']).median()
ratio_A = median_arm[('A', 'test_A_label_steps')] / max(n_obs['test_A_label_steps'], 1)
ratio_B = median_arm[('B', 'test_B_coordinate_steps')] / max(n_obs['test_B_coordinate_steps'], 1)
verdict = ('label–coordinate discordance (no within-segment claim from test A)' if ratio_A >= DECISION_HIGH else
           'label-noise explanation rejected' if ratio_A < DECISION_LOW else 'partly explained by label noise')
summary = pd.Series({'coordinate': COORDINATE_LABEL,
                     'observed test A calls (beyond label steps)': n_obs['test_A_label_steps'],
                     'observed test B calls (beyond coordinate steps)': n_obs['test_B_coordinate_steps'],
                     'observed T_spatial calls': n_obs['T_spatial'],
                     'relabeled test A calls': ', '.join(str(int(observed_counts.loc['test_A_label_steps', p]))
                                                          for p in observed_counts.columns if p != 'species'),
                     'relabeled test B calls': ', '.join(str(int(observed_counts.loc['test_B_coordinate_steps', p]))
                                                          for p in observed_counts.columns if p != 'species'),
                     'arm A (truth = coordinate steps): median test A calls': median_arm[('A', 'test_A_label_steps')],
                     'arm A: median test B calls': median_arm[('A', 'test_B_coordinate_steps')],
                     'arm B (truth = label steps): median test A calls': median_arm[('B', 'test_A_label_steps')],
                     'arm B: median test B calls': median_arm[('B', 'test_B_coordinate_steps')],
                     'primary ratio (arm A test A ÷ observed test A)': ratio_A,
                     'arm B test B ÷ observed test B': ratio_B,
                     'label agreement, mouse (mean)': thresholds[thresholds.species.eq('mouse')]['label agreement'].mean(),
                     'label agreement, human (mean)': thresholds[thresholds.species.eq('human')]['label agreement'].mean(),
                     'decision': verdict}, name='value')
display(summary.to_frame())

z_max = simulated.groupby(['statistic', 'pathway_id']).z_joint.max().unstack('statistic')
per_pathway = obs_species.pivot_table(index='pathway_id', columns='statistic', values=['z_joint', 'called'])
per_pathway.columns = [f'{a} {b}' for a, b in per_pathway.columns]
per_pathway = per_pathway.join(z_max.add_prefix('max simulated z_joint ').rename(columns=str))
per_pathway['within-segment candidate (A and B)'] = (per_pathway['called test_A_label_steps'].astype(bool)
                                                     & per_pathway['called test_B_coordinate_steps'].astype(bool))
per_pathway['A exceeds every simulated z'] = (per_pathway['z_joint test_A_label_steps']
                                              > per_pathway['max simulated z_joint test_A_label_steps'])
per_pathway['B exceeds every simulated z'] = (per_pathway['z_joint test_B_coordinate_steps']
                                              > per_pathway['max simulated z_joint test_B_coordinate_steps'])
per_pathway.insert(0, 'name', per_pathway.index.str.split('::').str[1].str.replace(r' R-HSA-\d+$', '', regex=True))
per_pathway.to_csv(tables / 'per_pathway_beyond_steps.csv')
print('Within-segment candidates on this coordinate (tests A and B):',
      int(per_pathway['within-segment candidate (A and B)'].sum()))
display(per_pathway[per_pathway['within-segment candidate (A and B)']]
        [['name', 'z_joint test_A_label_steps', 'z_joint test_B_coordinate_steps',
          'A exceeds every simulated z', 'B exceeds every simulated z']].round(2))

## 7 · Across coordinates

Reads every coordinate run of this notebook that exists. The rule: a pathway is a within-segment
candidate when tests A and B both call it on SCF13 and on at least one other coordinate. Runs that
are missing are listed, not assumed.

In [ ]:
runs = {}
for folder in sorted((results_root / 'pt_pathway_continuous' / '57_beyond_steps').glob('*/tables/per_pathway_beyond_steps.csv')):
    runs[folder.parents[1].name] = pd.read_csv(folder, index_col=0)['within-segment candidate (A and B)']
print('Coordinate runs found:', sorted(runs))
if 'scf13' in runs and len(runs) > 1:
    cand = pd.DataFrame(runs).fillna(False).astype(bool)
    others = [c for c in cand if c != 'scf13']
    cand['SCF13 and at least one other'] = cand['scf13'] & cand[others].any(axis=1)
    cand.to_csv(output.parent / 'within_segment_candidates_across_coordinates.csv')
    print('Within-segment candidates across coordinates:', int(cand['SCF13 and at least one other'].sum()))
    display(cand[cand['SCF13 and at least one other']])

record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'coordinate_file': coordinate_file.name,
          'coordinate_column': COORDINATE_COLUMN, 'coordinate_label': COORDINATE_LABEL,
          'n_structures': int(len(position)), 'n_genes': len(genes), 'n_pathways': len(gene_sets),
          'summary': {k: (v if isinstance(v, str) else float(v)) for k, v in summary.items()}}
(output / 'run_record.json').write_text(json.dumps(record, indent=2))
summary.to_csv(tables / 'decision_summary.csv')